# Module 2: X-Ray Fracture Detection with DenseNet121

Binary classification pipeline: Fracture vs. No Fracture

- **Model**: DenseNet121 (ImageNet pretrained)
- **Dataset A**: Unstructured (fracture/ and normal/ folders - manual split into train/val/test)
- **Dataset B**: Structured (train/, val/, test/ folders with fractured/ and not fractured/ subfolders)
- **Training**: Two-phase fine-tuning with early stopping
- **Device**: CUDA if available, CPU fallback

## Kaggle Datasets
- Dataset A: https://www.kaggle.com/datasets/bmadushanirodrigo/fracture-multi-region-x-ray-data
- Dataset B: https://www.kaggle.com/datasets/orvile/bone-fracture-dataset

## GitHub Repository
This notebook automatically clones the required modules from:
- **Branch**: `feat/module2`
- **Repo**: https://github.com/MaroSalah02/X-CV
- **Path**: Model Fine-tuning/Models Workspaces/X-Ray Fracture Detection

**Cell 1** will handle the GitHub setup automatically. Just run it first!

## How to Add Your GitHub Token to Kaggle

For accessing **private GitHub repositories**, you need to add your GitHub Personal Access Token to Kaggle Secrets:

### Step 1: Create GitHub Personal Access Token
1. Go to https://github.com/settings/tokens
2. Click **"Generate new token"** → **"Generate new token (classic)"**
3. Name it: `X-CV-Kaggle`
4. Select scopes: ✅ `repo` (full control of private repositories)
5. Click **"Generate token"**
6. **Copy the token** (you won't see it again!)

### Step 2: Add Token to Kaggle Notebook
1. Open this notebook in Kaggle
2. Click **⚙️ Settings** (top right)
3. Go to **Add-ons** → **Secrets**
4. Click **+ Add new secret**
5. Key: `GITHUB_TOKEN`
6. Value: **Paste your token**
7. Click **Save**

### Step 3: Run the Notebook
- Cell 1 will automatically load the token from Kaggle Secrets
- The notebook will clone the private repository
- All modules will be imported and ready to use

### ⚠️ Security Notes
- Never commit your token to GitHub
- Tokens are stored encrypted in Kaggle
- You can regenerate tokens anytime at https://github.com/settings/tokens
- The token is only used within this notebook

In [ ]:
# ============ Setup: Clone Private GitHub Repository ============
import subprocess
import os
import sys
from urllib.parse import urljoin

# ============ GitHub Token Setup ============
# In Kaggle, use Secrets (Add-ons → Secrets → add your GitHub token as 'GITHUB_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    print("✓ GitHub token loaded from Kaggle Secrets")
except Exception as e:
    print(f"⚠ Could not load from Kaggle Secrets: {e}")
    print("  You can add your token via: Add-ons → Secrets → Add 'GITHUB_TOKEN'")
    github_token = None

# ============ Repository Configuration ============
repo_url = "https://github.com/MaroSalah02/X-CV.git"
branch = "feat/module2"
module_dir = "Model Fine-tuning/Models Workspaces/X-Ray Fracture Detection"
clone_dir = "x-cv-repo"

# ============ Clone Repository ============
print("\n" + "="*70)
print("Cloning GitHub Repository")
print("="*70)

try:
    if github_token:
        # Use token for authentication (for private repos)
        auth_url = repo_url.replace("https://", f"https://{github_token}@")
        clone_cmd = ["git", "clone", "--branch", branch, "--depth", "1", auth_url, clone_dir]
        print(f"Using GitHub token for authentication...")
    else:
        # Public repo or token not provided
        clone_cmd = ["git", "clone", "--branch", branch, "--depth", "1", repo_url, clone_dir]
        print(f"Cloning without authentication (public repo)...")
    
    print(f"Repository: {repo_url}")
    print(f"Branch: {branch}")
    
    result = subprocess.run(
        clone_cmd,
        check=True,
        capture_output=True,
        text=True
    )
    print("✓ Repository cloned successfully")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Git clone failed!")
    print(f"Error: {e.stderr}")
    print("\nTROUBLESHOOTING:")
    print("1. Make sure your GitHub token has 'repo' scope access")
    print("2. Add token via Kaggle: Add-ons → Secrets → Add 'GITHUB_TOKEN'")
    print("3. Or uncomment the alternative setup in Cell 2")
    raise

# ============ Setup Python Path ============
module_path = os.path.join(clone_dir, module_dir)

if not os.path.exists(module_path):
    print(f"✗ Module path not found: {module_path}")
    print(f"Available paths: {os.listdir(clone_dir)}")
    raise FileNotFoundError(f"Module directory not found at {module_path}")

if module_path not in sys.path:
    sys.path.insert(0, module_path)

print(f"✓ Module path added: {module_path}")

# ============ Standard Imports ============
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

# Set seaborn style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# ============ Import Custom Modules ============
print("\nImporting custom modules...")
try:
    from data_preprocessing import (
        get_combined_dataloader,
        show_batch,
        get_class_weights
    )
    from model import (
        build_densenet121,
        unfreeze_layers,
        get_model_summary
    )
    from train import (
        train_model,
        plot_training_history,
        load_model
    )
    from evaluate_module2 import (
        evaluate_on_test,
        predict_single_image
    )
    print("✓ All custom modules imported successfully")
except ImportError as e:
    print(f"✗ Import error: {e}")
    raise

print("\n" + "="*70)
print("Environment Information")
print("="*70)
print(f"PyTorch version: {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*70)
print("✓ Setup completed successfully!\n")

In [ ]:
# ============ Troubleshooting: Alternative Setup Methods ============
# Use this ONLY if Cell 1 failed

"""
import urllib.request
import zipfile
import shutil
import base64

# Method 1: Using GitHub Token with ZIP Download
def setup_from_github_zip_with_token(github_token):
    '''Download private repo as ZIP using GitHub token'''
    # GitHub API to get archive link
    owner = "MaroSalah02"
    repo = "X-CV"
    branch = "feat/module2"
    
    zip_url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{branch}.zip"
    zip_path = "x-cv-repo.zip"
    
    print("Downloading repository as ZIP...")
    
    # Create auth header
    headers = {'Authorization': f'token {github_token}'}
    request = urllib.request.Request(zip_url, headers=headers)
    
    try:
        with urllib.request.urlopen(request) as response:
            with open(zip_path, 'wb') as out_file:
                out_file.write(response.read())
        print("✓ Repository downloaded")
    except Exception as e:
        print(f"✗ Download failed: {e}")
        return False
    
    print("Extracting repository...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(".")
        
        # Find and setup the module path
        module_base = f"X-CV-{branch}/Model Fine-tuning/Models Workspaces/X-Ray Fracture Detection"
        if os.path.exists(module_base):
            sys.path.insert(0, module_base)
            print(f"✓ Module path added: {module_base}")
            os.remove(zip_path)
            return True
        else:
            print(f"✗ Module path not found at: {module_base}")
            return False
    except Exception as e:
        print(f"✗ Extraction failed: {e}")
        return False

# Method 2: Using Git with SSH (if you have SSH key configured)
def setup_from_github_ssh():
    '''Clone using SSH key (requires SSH key setup in Kaggle)'''
    repo_url = "git@github.com:MaroSalah02/X-CV.git"
    branch = "feat/module2"
    clone_dir = "x-cv-repo"
    
    try:
        subprocess.run(
            ["git", "clone", "--branch", branch, "--depth", "1", repo_url, clone_dir],
            check=True,
            capture_output=True
        )
        print("✓ Repository cloned via SSH")
        return True
    except subprocess.CalledProcessError as e:
        print(f"✗ SSH clone failed: {e.stderr}")
        return False

# UNCOMMENT AND RUN ONE OF THESE IF CELL 1 FAILED:

# Option A: If you have the token available here
# github_token = "your_token_here"  # Replace with your actual token
# success = setup_from_github_zip_with_token(github_token)

# Option B: If you have SSH configured
# success = setup_from_github_ssh()

print("✓ Troubleshooting options available. Uncomment one above if needed.")
"""

print("✓ Troubleshooting cell loaded. Use only if Cell 1 fails.")

In [ ]:
# ============ Kaggle Configuration ============
# Update these paths based on your Kaggle environment
ROOT_A = '/kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification'
ROOT_B = '/kaggle/input/bone-fracture-dataset/Bone fracture dataset/Bone fracture dataset/Dataset'

# For local development, use:
# ROOT_A = './Bone_Fracture_Binary_Classification'
# ROOT_B = './Bone fracture dataset'

BATCH_SIZE = 32
NUM_EPOCHS = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = './weights'

# Create weights directory
os.makedirs(SAVE_DIR, exist_ok=True)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True

print("="*70)
print("Configuration")
print("="*70)
print(f"Dataset A root: {ROOT_A}")
print(f"Dataset B root: {ROOT_B}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Save directory: {SAVE_DIR}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*70)

In [ ]:
# Load dataloaders for all phases
print("Loading dataloaders...")
train_loader = get_combined_dataloader(
    ROOT_A, ROOT_B,
    phase='train',
    batch_size=BATCH_SIZE
)
val_loader = get_combined_dataloader(
    ROOT_A, ROOT_B,
    phase='val',
    batch_size=BATCH_SIZE
)
test_loader = get_combined_dataloader(
    ROOT_A, ROOT_B,
    phase='test',
    batch_size=BATCH_SIZE
)

print("\n" + "="*70)
print("Dataset Information")
print("="*70)
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Total training samples: {len(train_loader) * BATCH_SIZE}")
print(f"Total validation samples: {len(val_loader) * BATCH_SIZE}")
print(f"Total test samples: {len(test_loader) * BATCH_SIZE}")
print("="*70)

# Visualize a batch
print("\nVisualizing training batch...")
show_batch(train_loader, n=8)

In [ ]:
# Build model
print("Building DenseNet121 model...")
model = build_densenet121(
    num_classes=2,
    pretrained=True,
    dropout_rate=0.5
)
model = model.to(DEVICE)

# Print model summary
get_model_summary(model)

print("\nModel architecture (first 5 layers):")
for i, (name, module) in enumerate(model.named_children()):
    if i < 5:
        print(f"  {name}: {module.__class__.__name__}")

In [ ]:
# Train the model
print("\nStarting training...\n")
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
    save_dir=SAVE_DIR,
    patience=7
)

print("\nTraining completed!")

In [ ]:
# Plot training curves
print("Plotting training history...")
plot_training_history(history)

# Print final metrics
print("\n" + "="*70)
print("Final Training Metrics")
print("="*70)
print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")
print(f"Final training loss: {history['train_loss'][-1]:.4f}")
print(f"Final validation loss: {history['val_loss'][-1]:.4f}")
print("="*70)

In [ ]:
# Load best model
best_model = load_model(
    checkpoint_path=os.path.join(SAVE_DIR, 'densenet121_best.pth'),
    device=DEVICE
)

# Evaluate on test set
evaluate_on_test(best_model, test_loader, DEVICE)

In [ ]:
# Example: Single image prediction
# Replace 'sample_xray.jpg' with actual image path
sample_image_path = './sample_xray.jpg'

# Check if sample image exists
if os.path.exists(sample_image_path):
    print(f"Predicting on: {sample_image_path}\n")
    result = predict_single_image(
        best_model,
        sample_image_path,
        device=DEVICE
    )
    
    print("="*70)
    print("Prediction Result")
    print("="*70)
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"\nProbabilities:")
    for class_name, prob in result['probabilities'].items():
        print(f"  {class_name}: {prob:.4f}")
    print("="*70)
else:
    print(f"Sample image not found at: {sample_image_path}")
    print("To test single image prediction, provide a valid image path.")
    print("\nExample usage:")
    print("  result = predict_single_image(best_model, 'path/to/image.jpg', device=DEVICE)")
    print("  print(result)")